In [1]:
#%pip install pyserial

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


import serial
import threading
import time


### Loading the Model

In [3]:
# Train REGULARIZED version
class EmotionCNN_Reg(nn.Module):
    def __init__(self, input_channels=20, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv1d(input_channels, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        self.fc1       = nn.Linear(32 * 31, 64)
        self.dropout1  = nn.Dropout(0.3)
        self.fc2       = nn.Linear(64, 32)
        self.dropout2  = nn.Dropout(0.5)
        self.fc3       = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.dropout(x, 0.2)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

# Retrain with regularization
'''model_reg = EmotionCNN_Reg(input_channels=X_train_torch.shape[1],
                           num_classes=4).to(device)
optimizer_reg = optim.Adam(model_reg.parameters(), lr=0.001, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

print("🛡️ Training regularized model...")
print("Input channels:", X_train_torch.shape[1])
print("Total params:", sum(p.numel() for p in model_reg.parameters()))
'''

'model_reg = EmotionCNN_Reg(input_channels=X_train_torch.shape[1],\n                           num_classes=4).to(device)\noptimizer_reg = optim.Adam(model_reg.parameters(), lr=0.001, weight_decay=1e-3)\ncriterion = nn.CrossEntropyLoss()\n\nprint("🛡️ Training regularized model...")\nprint("Input channels:", X_train_torch.shape[1])\nprint("Total params:", sum(p.numel() for p in model_reg.parameters()))\n'

In [4]:
# Load the best CNN model

PATH= 'best_emotion_cnn_reg_125_windowsize.pth'

model = EmotionCNN_Reg()

model.load_state_dict(torch.load(PATH, weights_only=True))
model.eval()


EmotionCNN_Reg(
  (conv1): Conv1d(20, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=992, out_features=64, bias=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=32, out_features=4, bias=True)
)

### Live Data Streaming

In [5]:
'''if torch.backends.mps.is_available():
    device = torch.device("mps")  

else:
    device = torch.device("cpu")''' 
device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [6]:
# Start Data streaming
# How is the live data coming in?

COM_PORT='/dev/cu.usbmodem21301'
# ser = serial.Serial(COM_PORT, 115200) 

window_size = 125
overlap_size = int(0.25 * window_size)
step_size = window_size - overlap_size # 94

live_buffer = []               # The shared "bowl"
buffer_lock = threading.Lock() # The "Pause" button to prevent data crashes

def data_collection_thread():
    # Open the serial port inside the producer
    ser = serial.Serial(COM_PORT, 115200)
    ser.reset_input_buffer() # Clear out old junk!
    
    correct_columns=21


    print("Producer: Listening to Arduino at 30Hz...")
    
    while True:
        try:
            # new code to check if the line has the correct number of columns before next step
            raw_line = ser.readline().decode('utf-8').strip()
            sample_list=[]

            indi_params= raw_line.split(',')
            if len(indi_params) == correct_columns:
                sample_list = [float(x) for x in indi_params] #changed this up to make sure its not a generator and its a list
                with buffer_lock:
                    live_buffer.append(sample_list)
            else:
                pass

            # previous code that takes number of columns as assumption
            '''for x in raw_line.split(','):
                #print("This should be values within the line starting with host time: ", x)
                sample_list.append(float(x))
            
            #row_data = [float(x) for x in raw_line.split(',')]
            #print("Raw Line: ", raw_line) # debug
            #raw_data=row_data[0]
            #print(raw_data)# debug
            #row_data.append(sample_list) # Append the list of values as a single row in the buffer
            
            # 2. Lock the buffer, add the data, unlock
            with buffer_lock:
                live_buffer.append(sample_list)'''
                
        except Exception as e:
            pass


def inference_thread():
    class_labels = {0: "Distracted", 1: "Focus", 2: "Relaxed", 3: "Stress"}
    
    print("Consumer: Waiting for 125 rows...")
    model.eval() 
    
    while True:
        data_to_process = None
        
        # Pull from buffer
        with buffer_lock:
            if len(live_buffer) >= window_size:
                data_to_process = list(live_buffer[:window_size])
                del live_buffer[:step_size] 
        
        if data_to_process is not None:
            # Convert to numpy (Shape: 125, 21)
            data_array = np.array(data_to_process,dtype=np.float32)
            
            data_array = data_array[:, 1:] 

            mean = np.mean(data_array, axis=0)
            std = np.std(data_array, axis=0)
            data_normalized = (data_array - mean) / (std + 1e-8)

            input_tensor = torch.tensor(data_normalized, dtype=torch.float32)#Convert to Torch Tensor
            
            input_tensor = input_tensor.transpose(0, 1)
            
            input_tensor = input_tensor.unsqueeze(0).to(device)# making it (1, Channels, 125) 

            try:
                with torch.no_grad():
                    output = model(input_tensor)
                    
                    probabilities = torch.softmax(output, dim=1)
                    
                    conf, pred_idx = torch.max(probabilities, dim=1)
                    
                    prediction = pred_idx.item()
                    confidence = conf.item()

                print(f"Result: {class_labels.get(prediction, 'Unknown')} | Conf: {confidence:.2%}")
                
            except Exception as e:
                print(f"Inference Error: {e}")
                print(f"Check if input_tensor shape {input_tensor.shape} matches model!")

        else:
            time.sleep(0.01)
# The thread will check the buffer every 10ms, and only do ML math when we have a full window of 125 rows = old inference thread.
'''def inference_thread():
    print("Consumer: Waiting for 125 rows...")
    while True:
        data_to_process = None
        # 1. Safely check the buffer size
        with buffer_lock:
            if len(live_buffer) >= window_size:
                # Copy the first 125 rows for PyTorch
                data_to_process = live_buffer[:window_size]
                
                # The Slide: Delete the oldest 94 rows IN PLACE, leaving the 31 overlap
                del live_buffer[:step_size] 
        
        # 2. Do the heavy ML math OUTSIDE the lock!
        if data_to_process is not None:
            print("Window extracted! Buffer safely sliced.")
            print("Data to Process: ",data_to_process)
        
            # Normalize the data
            data_to_process = np.array(data_to_process)
            mean = np.mean(data_to_process, axis=0)
            std = np.std(data_to_process, axis=0)
            data_to_process = np.subtract(data_to_process, mean) / (std + 1e-8)
            print("Data Normalized", data_to_process.shape)

            with torch.no_grad():
                input_tensor = torch.tensor(data_to_process, dtype=torch.float32)
                output = model(input_tensor)
            print(f"Output: {output}")

            # model(data_to_process)
            # print("Prediction: Stressed!")
            
        else:
            # If we don't have 125 rows yet, sleep for a tiny fraction of a second 
            # so we don't max out the computer's CPU while waiting.
            time.sleep(0.01)
'''



# Create the worker threads
t1 = threading.Thread(target=data_collection_thread, daemon=True)
t2 = threading.Thread(target=inference_thread, daemon=True)

# Start them
t1.start()
t2.start()

# Keep the main script alive so the threads can run in the background
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down the system...")




Consumer: Waiting for 125 rows...
Producer: Listening to Arduino at 30Hz...
Result: Focus | Conf: 95.90%
Result: Stress | Conf: 82.86%
Result: Stress | Conf: 99.99%
Result: Stress | Conf: 99.95%
Result: Stress | Conf: 97.07%
Result: Distracted | Conf: 61.65%
Result: Focus | Conf: 41.07%
Result: Distracted | Conf: 94.87%
Result: Distracted | Conf: 83.07%
Result: Relaxed | Conf: 99.85%
Result: Stress | Conf: 74.13%
Result: Distracted | Conf: 92.72%
Shutting down the system...


Result: Distracted | Conf: 49.88%
Result: Distracted | Conf: 60.37%
Result: Relaxed | Conf: 96.72%
Result: Relaxed | Conf: 73.79%
Result: Focus | Conf: 95.83%
Result: Focus | Conf: 67.02%
Result: Focus | Conf: 56.42%
Result: Relaxed | Conf: 49.40%
Result: Focus | Conf: 60.25%
Result: Focus | Conf: 61.87%
Result: Focus | Conf: 75.38%
Result: Focus | Conf: 99.72%
Result: Focus | Conf: 95.32%
Result: Focus | Conf: 74.65%
Result: Focus | Conf: 52.37%
Result: Focus | Conf: 46.94%
Result: Relaxed | Conf: 58.54%
Result: Focus | Conf: 47.79%
Result: Focus | Conf: 75.59%
Result: Stress | Conf: 98.85%
Result: Distracted | Conf: 37.75%
Result: Distracted | Conf: 73.43%
Result: Stress | Conf: 37.23%
Result: Relaxed | Conf: 37.89%
Result: Focus | Conf: 57.79%
Result: Relaxed | Conf: 59.43%
Result: Relaxed | Conf: 56.96%
Result: Focus | Conf: 57.33%
Result: Focus | Conf: 45.51%
Result: Focus | Conf: 31.66%
Result: Focus | Conf: 38.82%
Result: Focus | Conf: 74.96%
Result: Relaxed | Conf: 52.52%
Resul

### Data Processing

In [ ]:
# Call Data Processing Pipeline

### Model Prediction

In [ ]:
# Live Predictions

